# NoClip · 2DGS 실험 (Google Colab)

줄무늬의 원인은 스플랫이 카메라 방향으로 길게 늘어나는 것. **2DGS** 는 스플랫을 납작한 디스크로 제한하고
깊이·법선 손실을 걸어서 이런 늘어남을 구조적으로 막는 방식임. 공식 구현: `hbb1/2d-gaussian-splatting`.

**준비** — `difix_colab.ipynb` 와 같은 데이터 폴더를 그대로 씀
1. `F:\Sample\6\1b\data` 안의 `images`, `sparse` 를 `G:\내 드라이브\3DGS\colab\corridor1` 에 복사 (이미 했으면 생략)
2. **런타임 → 런타임 유형 변경** 에서 GPU (L4 가 보이면 L4)

**순서**: **런타임 → 모두 실행**. 결과(학습 결과, PLY)는 드라이브 `3DGS/colab/out/<방>/gs2d` 에 저장됨.

**알아둘 점**
- 2DGS PLY 는 크기 값이 2개뿐이라 SuperSplat/Unity 에서 바로 안 보임 → 마지막 셀이 3DGS 형식으로 변환함
- 라이선스는 원본 3DGS 와 같은 비상업 연구용
- 걸리는 시간은 아직 측정 전. 처음 돌린 뒤 README 에 기록할 것

## 1. 드라이브 연결 · 설정

In [ ]:
# GPU 가 아니면 여기서 멈춤 (CPU 런타임에서는 CUDA 컴파일이 불가능하고, 설치만 오래 걸린 뒤 실패함)
import subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise SystemExit("GPU 런타임이 아님 -> 런타임 > 런타임 유형 변경 에서 T4/L4 선택. 선택지가 없으면 무료 GPU 한도 소진 상태이니 나중에 다시 시도")
from google.colab import drive
drive.mount('/content/drive')

SCENE      = "corridor1"                                   # 드라이브 3DGS/colab/ 아래 폴더 이름
DRIVE_DATA = f"/content/drive/MyDrive/3DGS/colab/{SCENE}"
OUT        = f"/content/drive/MyDrive/3DGS/colab/out/{SCENE}/gs2d"
DATA       = f"/content/data/{SCENE}"                  # 빠른 로컬 복사본
FACTOR     = 4      # 4K 원본 기준 4 -> 960x540. 이미지가 메모리(RAM)에 올라가서 2 로 하면 복도(1000장 이상)에서 터질 수 있음
SH         = 2      # Brush 와 동일
ITERS      = 30000
LAMBDA_NORMAL = 0.05   # 2DGS 기본값
LAMBDA_DIST   = 0.0    # 깊이 뭉침 벌점. 기본 0 (넓은 장면 권장). 올리면 얇아지지만 번지는 걸 막을 수 있음

import os
os.environ.update(SCENE=SCENE, DRIVE_DATA=DRIVE_DATA, OUT=OUT, DATA=DATA, FACTOR=str(FACTOR), SH=str(SH),
                  ITERS=str(ITERS), LN=str(LAMBDA_NORMAL), LD=str(LAMBDA_DIST))
os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!ls "$DRIVE_DATA"

## 2. 설치 · CUDA 컴파일 (10~20분)
저장소는 `--recursive` 로 받아야 함 (안에 하위 모듈이 있음).

In [ ]:
%cd /content
import os
if not os.path.exists("/content/2d-gaussian-splatting/submodules/diff-surfel-rasterization/setup.py"):
    !rm -rf 2d-gaussian-splatting
    !git clone -q --recursive https://github.com/hbb1/2d-gaussian-splatting.git
%cd /content/2d-gaussian-splatting

# simple-knn 은 gitlab 에서 받는데 막히면 GitHub 미러로 대체
if not os.path.exists("submodules/simple-knn/simple_knn.cu"):
    !rm -rf submodules/simple-knn
    !git clone -q --recursive https://github.com/camenduru/simple-knn.git submodules/simple-knn

# 새 컴파일러에서 나는 알려진 오류 방지용 include 추가 (이미 있으면 건너뜀)
!grep -q "<cstdint>" submodules/diff-surfel-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' submodules/diff-surfel-rasterization/cuda_rasterizer/rasterizer_impl.h
!grep -q "<cfloat>" submodules/simple-knn/simple_knn.cu || sed -i '1i #include <cfloat>' submodules/simple-knn/simple_knn.cu

!pip install -q plyfile tqdm opencv-python lpips ninja
# T4 = 7.5, L4 = 8.9
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.9"
# 실패하면 원인이 보이도록 -q 없이 마지막 줄만 출력. 런타임이 새로 켜지면 이 셀부터 다시 실행해야 함
!pip install --no-build-isolation submodules/diff-surfel-rasterization 2>&1 | tail -25
!pip install --no-build-isolation submodules/simple-knn 2>&1 | tail -25
!python -c "import diff_surfel_rasterization, simple_knn; print('컴파일 OK')"

## 3. 데이터 복사 (드라이브 → 로컬)
`images/` 와 `sparse/0` 구조면 그대로 읽힘 (Brush 에 넣던 `Nb\data` 와 같은 형식).

In [ ]:
import os, shutil
# DRIVE_DATA 안에 images/, sparse/ 가 있어야 함. 다른 위치에 올렸으면 1번 셀의 DRIVE_DATA 를 그 폴더로 바꿀 것
shutil.rmtree(DATA, ignore_errors=True)
os.makedirs(DATA)
shutil.copytree(f"{DRIVE_DATA}/images", f"{DATA}/images")
shutil.copytree(f"{DRIVE_DATA}/sparse", f"{DATA}/sparse")
print("sparse:", os.listdir(f"{DATA}/sparse"), "images:", len(os.listdir(f"{DATA}/images")))


## 4. 2DGS 학습
30,000 스텝. 10,000 / 20,000 스텝마다 중간 결과도 저장.

In [ ]:
%cd /content/2d-gaussian-splatting
!python train.py -s "$DATA" -m "$OUT/model" -r $FACTOR --sh_degree $SH --iterations $ITERS \
    --lambda_normal $LN --lambda_dist $LD --depth_ratio 0 \
    --save_iterations 10000 20000 $ITERS --test_iterations -1

## 5. PLY 변환 · 저장
2DGS 형식 → SuperSplat/Unity 가 읽는 3DGS 형식 (세 번째 크기를 아주 작게 채움).

In [ ]:
%%writefile /content/nc_ply2d_to_3d.py
import sys
import numpy as np

def read_ply(path):
    with open(path, "rb") as f:
        names, count = [], 0
        while True:
            line = f.readline().decode("ascii").strip()
            if line.startswith("element vertex"):
                count = int(line.split()[-1])
            elif line.startswith("property"):
                names.append(line.split()[-1])
            elif line == "end_header":
                break
        data = np.frombuffer(f.read(count * len(names) * 4), dtype="<f4").reshape(count, len(names))
    return names, data

src, dst = sys.argv[1], sys.argv[2]
thick = float(sys.argv[3]) if len(sys.argv) > 3 else -14.0
names, data = read_ply(src)
if "scale_2" in names:
    out_names, out = names, data
else:
    i = names.index("scale_1") + 1
    out_names = names[:i] + ["scale_2"] + names[i:]
    out = np.concatenate([data[:, :i], np.full((len(data), 1), thick, dtype="<f4"), data[:, i:]], axis=1)
with open(dst, "wb") as f:
    head = "ply\nformat binary_little_endian 1.0\nelement vertex %d\n" % len(out)
    head += "".join("property float %s\n" % n for n in out_names) + "end_header\n"
    f.write(head.encode("ascii"))
    f.write(np.ascontiguousarray(out, dtype="<f4").tobytes())
print(dst, len(out), "splats")

In [ ]:
import os, glob
os.makedirs(f"{OUT}/ply", exist_ok=True)
for it in sorted(glob.glob(f"{OUT}/model/point_cloud/iteration_*")):
    n = it.split("_")[-1]
    !python /content/nc_ply2d_to_3d.py "{it}/point_cloud.ply" "{OUT}/ply/{SCENE}_2dgs_{n}.ply"
!ls -la "$OUT/ply"